# Reading directly from the volume to writing data into tables
Source data is read into the bronze layer without any tranformations

In [0]:
#List of source data directories
sourcePath = ["/Volumes/data_analytics/bronzelayer/rawdata/source_crm/", "/Volumes/data_analytics/bronzelayer/rawdata/source_erp/"]

for folder in sourcePath:
 
    try:
        files = dbutils.fs.ls(folder)
    except Exception as e:
        print(f"Skipping path {folder} - not found or inaccessible.")
        continue

    for file in files:
        
        if file.name.endswith(".csv"):

            file_path = file.path
            clean_file_name = file.name.replace(".csv", "").replace("-", "_").lower()
        
            print(f"Processing: {file.name} -> Target Table: {clean_file_name}")
        
            df = (spark.read
                  .format("csv")
                  .option("header", "true")
                  .option("inferSchema", "true")
                  .load(file_path))
            
            (df.write
             .format("delta")
             .mode("overwrite")
             .saveAsTable(clean_file_name))